# 3. Submission - XGBoost con Feature Engineering

Cargar el modelo XGBoost con feature engineering, reentrenar sobre todo el train, predecir sobre test y enviar a Kaggle.

**Modelo:** XGBoost con hiperparámetros optimizados + 11 features generadas (24 total)

**Competencia:** `playground-series-s6e2`

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import joblib

## 2. Carga de datos

In [ ]:
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test.csv")

# Target binario
train["target"] = (train["Heart Disease"] == "Presence").astype(int)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

## 3. Feature Engineering

Misma función `build_features()` que en el notebook de modelo.
Se aplica tanto a train como a test para garantizar consistencia.

In [ ]:
def build_features(df):
    """Aplica feature engineering sobre un DataFrame con las features originales.
    Retorna el DataFrame con las features nuevas agregadas (no modifica el original).
    """
    X = df.copy()

    # ── Grupo A: Indicadores clínicos derivados ──────────────────────────────
    X["HR_reserve"] = X["Max HR"] - X["Age"]
    X["Double_Product"] = X["BP"] * X["Max HR"]
    X["Chol_HR_ratio"] = X["Cholesterol"] / (X["Max HR"] + 1)

    # ── Grupo B: Interacciones ───────────────────────────────────────────────
    X["Age_x_MaxHR"] = X["Age"] * X["Max HR"]
    X["BP_x_Chol"] = X["BP"] * X["Cholesterol"]
    X["Age_x_STdep"] = X["Age"] * X["ST depression"]

    # ── Grupo C: Transformaciones no lineales ────────────────────────────────
    X["ST_dep_sq"] = X["ST depression"] ** 2
    X["ST_dep_present"] = (X["ST depression"] > 0).astype(int)

    # ── Grupo D: Binning clínico ─────────────────────────────────────────────
    X["Age_group"] = pd.cut(
        X["Age"],
        bins=[0, 44, 54, 64, 120],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X["BP_category"] = pd.cut(
        X["BP"],
        bins=[0, 119, 129, 139, 300],
        labels=[0, 1, 2, 3]
    ).astype(int)

    X["Chol_category"] = pd.cut(
        X["Cholesterol"],
        bins=[0, 199, 239, 1000],
        labels=[0, 1, 2]
    ).astype(int)

    return X


# Aplicar FE a train y test
train_fe = build_features(train)
test_fe = build_features(test)

print(f"Train FE: {train_fe.shape}")
print(f"Test FE:  {test_fe.shape}")

## 4. Cargar modelo y reentrenar sobre todo el train

In [ ]:
saved = joblib.load("../../models/3_xgboost_fe_best.pkl")
best_params = saved["best_params"]
model_features = saved["model_features"]
cv_score = saved["cv_score"]
cv_std = saved["cv_std"]

print(f"Hiperparámetros: {best_params}")
print(f"Features ({len(model_features)}): {model_features}")
print(f"ROC-AUC (CV): {cv_score:.4f} +/- {cv_std:.4f}")

# Reentrenar sobre todo el dataset de train
X_train = train_fe[model_features]
y_train = train_fe["target"]

model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    **best_params
)
model.fit(X_train, y_train, verbose=False)
print(f"\nModelo entrenado sobre {len(X_train)} muestras.")

## 5. Predicción sobre test

In [ ]:
X_test = test_fe[model_features]
test_probs = model.predict_proba(X_test)[:, 1]

# Submission con probabilidades (formato requerido por Kaggle)
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": np.round(test_probs, 4)
})

print(f"Shape: {submission.shape}")
print(f"\nEstadísticas de probabilidades:")
print(submission["Heart Disease"].describe().round(4))
print(f"\nPrimeras filas:")
submission.head(10)

## 6. Guardar CSV

In [ ]:
SUBMISSION_FILE = "3_XGB_FE_submission.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

# Verificar
check = pd.read_csv(SUBMISSION_FILE)
print(f"Archivo: {SUBMISSION_FILE}")
print(f"Shape: {check.shape}")
print(f"Columnas: {list(check.columns)}")
print(f"IDs: {check['id'].min()} - {check['id'].max()}")
check.head()

## 7. Submit a Kaggle

In [ ]:
COMPETITION = "playground-series-s6e2"
MESSAGE = "XGBoost FE - 24 features (13 originales + 11 generadas) - RandomizedSearchCV"

!kaggle competitions submit -c {COMPETITION} -f {SUBMISSION_FILE} -m "{MESSAGE}"